In [1]:
import pandas as pd
df = pd.read_csv("Hydrogen_Uncertainty_dataset.csv")

In [2]:
df

,"Natural Gas Cost ($/MMBtu, HHV)",Absorber CO2 Removal Efficiency (%),Capacity Factor (%),Performance Parameter: Annual Operating Hours (hours),Performance Parameter: Annual H2 Production (million tonne/yr),Performance Parameter: Annual CO2 Captured (million tonne/yr),Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2),Variable O&M Cost: CO2 T&S: O&M Cost ($/lb H2),CO2 Captured (tons/hr),Cost of CO2 Captured ($/ton)
0,7.975000,96.3675,76.5375,6709.276855,0.125799,1.284130,1.184908,0.065090,210.978004,232.157446
1,10.424999,95.7825,73.5625,6448.488770,0.120909,1.231241,1.412497,0.064933,210.469291,277.417550
2,7.825000,97.4025,92.6375,8120.603027,0.152261,1.560883,1.118635,0.065368,211.877999,218.241634
3,3.325000,94.2075,61.4875,5389.994141,0.101062,1.022440,0.850518,0.064511,209.099729,168.137687
4,8.525001,90.3375,84.0625,7368.918945,0.138167,1.375331,1.198089,0.063472,205.734506,240.722633
...,...,...,...,...,...,...,...,...,...,...
195,2.425000,96.4575,84.5875,7414.940430,0.139030,1.419718,0.666642,0.065114,211.056248,130.565850
196,7.675000,94.7475,87.5625,7675.728516,0.143920,1.459296,1.117000,0.064655,209.569287,220.323344
197,4.275000,98.7075,87.0375,7629.707031,0.143057,1.474381,0.828909,0.065718,213.012786,160.855549
198,2.675000,94.4775,66.9125,5865.549316,0.109979,1.113899,0.761790,0.064583,209.334506,150.428225


In [3]:
df.isna().sum()

Natural Gas Cost ($/MMBtu, HHV)                                   0
Absorber CO2 Removal Efficiency (%)                               0
Capacity Factor (%)                                               0
Performance Parameter: Annual Operating Hours (hours)             0
Performance Parameter: Annual H2 Production (million tonne/yr)    0
Performance Parameter: Annual CO2 Captured (million tonne/yr)     0
Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2)    0
Variable O&M Cost: CO2 T&S: O&M Cost ($/lb H2)                    0
CO2 Captured (tons/hr)                                            0
Cost of CO2 Captured ($/ton)                                      0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
x = df[['Natural Gas Cost ($/MMBtu, HHV)',
    'Absorber CO2 Removal Efficiency (%)',
    'Capacity Factor (%)']]

y = df['Performance Parameter: Annual H2 Production (million tonne/yr)']

In [6]:
x

,"Natural Gas Cost ($/MMBtu, HHV)",Absorber CO2 Removal Efficiency (%),Capacity Factor (%)
0,7.975000,96.3675,76.5375
1,10.424999,95.7825,73.5625
2,7.825000,97.4025,92.6375
3,3.325000,94.2075,61.4875
4,8.525001,90.3375,84.0625
...,...,...,...
195,2.425000,96.4575,84.5875
196,7.675000,94.7475,87.5625
197,4.275000,98.7075,87.0375
198,2.675000,94.4775,66.9125


In [7]:
y

0      0.125799
1      0.120909
2      0.152261
3      0.101062
4      0.138167
         ...   
195    0.139030
196    0.143920
197    0.143057
198    0.109979
199    0.152549
Name: Performance Parameter: Annual H2 Production (million tonne/yr), Length: 200, dtype: float64

In [8]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== PYOMO MODEL ===\n")


=== PYOMO MODEL ===



In [9]:
# Linear Regression
reg_pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('reg',LinearRegression())
])
reg_grid = GridSearchCV(reg_pipe, {}, cv=5, scoring='r2', n_jobs=-1)
reg_grid.fit(x_train,y_train)
print("Linear Regression best params:",reg_grid.best_params_)

best_reg = reg_grid.best_estimator_
print('Train R2:',best_reg.score(x_train,y_train))
print('Test R2:',best_reg.score(x_test,y_test))

Linear Regression best params: {}
Train R2: 0.999999999999851
Test R2: 0.9999999999998589


In [10]:
#Checking the RMSE and MAPE scores
from sklearn.metrics import mean_squared_error
import numpy as np

print("\n=== RMSE & MAPE ===\n")

models = {
    'Polynomial':        best_reg
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    print(f"{name} | RMSE: {rmse:.4f} | MAPE: {mape:.4f}%")


=== RMSE & MAPE ===

Polynomial | RMSE: 0.0000 | MAPE: 0.0000%


In [13]:
import json
import joblib
import numpy as np

scaler = best_reg.named_steps['scaler']
reg    = best_reg.named_steps['reg']
feature_names = list(x.columns)

coef_real      = reg.coef_ / scaler.scale_
intercept_real = reg.intercept_ - np.sum(reg.coef_ * scaler.mean_ / scaler.scale_)

equation = {
    "intercept": float(intercept_real),
    "coefficients": {feature_names[i]: float(coef_real[i]) for i in range(len(feature_names))}
}

def equation_value(eq, row):
    total = eq["intercept"]
    for feat, coef in eq["coefficients"].items():
        total += coef * row[feat]
    return total

check = np.array([equation_value(equation, r) for _, r in x.iterrows()])
assert np.allclose(check, best_reg.predict(x), atol=1e-6), "equation does not match model!"
print("equation matches pipeline ✓")

with open("Hydrogen_H2_Production_linear_eq.json", "w") as f:
    json.dump(equation, f, indent=2)
print("saved Hydrogen_H2_Production_linear_eq.json")

joblib.dump(best_reg, "Hydrogen_H2_Production_linear.pkl")
print("saved Hydrogen_H2_Production_linear.pkl")

equation matches pipeline ✓
saved Hydrogen_H2_Production_linear_eq.json
saved Hydrogen_H2_Production_linear.pkl
